# 01 — Limpieza previa de datos PlayerMaker

Notebook de comprobaciones iniciales sobre el dataset `PlayerMaker_Matrix.xlsx`.  
El objetivo es verificar la **consistencia interna** de los datos antes de realizar transformaciones.

## 1. Carga y exploración inicial

In [107]:
import pandas as pd
import numpy as np

# Carga del dataset
df = pd.read_excel("../Datos/PlayerMaker_Matrix.xlsx")
print(f"Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

Dimensiones: 5321 filas × 30 columnas


,Session Id,Phase Id,Phase Duration (min),Participated Players,Player Name,Position Category,Position,Player Participation Time (min),Total Touches (#),Left Leg Touches (#),...,SD Covered (m) [> 5.5(m/s)],Release Velocity Max (m/s),Release Velocity Avg (m/s),Release Index Beta - Total,Player Id,Agrupacion,Espacio,Polaridad,Equilibrio,NombreCorrecto
0,1656564,1656692,13,22,X X,Midfielders,Defensive Midfielder,13.0,33.0,11.0,...,2.0,18.4,11.8,11.8,111224,AGrande,EReducido,Polarizado,Equilibrio,Juvenil División de Honor
1,1656564,1656692,13,22,A. MADINA 2648,Forwards,Right Winger,13.0,21.0,9.0,...,6.0,16.6,13.2,6.6,113152,AGrande,EReducido,Polarizado,Equilibrio,Juvenil División de Honor
2,1656564,1656692,13,22,NORES 2109,Defenders,Right Center Back,13.0,34.0,14.0,...,5.0,16.2,10.9,14.2,113153,AGrande,EReducido,Polarizado,Equilibrio,Juvenil División de Honor
3,1656564,1656692,13,22,ETXEZARRETA 2230,Forwards,Right Winger,13.0,3.0,1.0,...,0.0,5.7,5.7,0.6,113155,AGrande,EReducido,Polarizado,Equilibrio,Juvenil División de Honor
4,1656564,1656692,13,22,GOROSABEL 1691,Midfielders,Center Midfielder,13.0,18.0,4.0,...,55.0,20.2,15.5,7.8,113159,AGrande,EReducido,Polarizado,Equilibrio,Juvenil División de Honor


## 2. Comprobaciones de consistencia sobre Phase Id

Cada `Phase Id` identifica una fase de entrenamiento/partido. Para todas las filas con el mismo
`Phase Id`, esperamos que:

1. **`Phase Duration (min)`** sea idéntico (la fase dura lo mismo para todos los jugadores).
2. **`Participated Players`** sea idéntico (el número de participantes no cambia entre filas de la misma fase).

### 2.1 ¿Es `Phase Duration (min)` consistente dentro de cada `Phase Id`?

Para cada `Phase Id` contamos cuántos valores **distintos** de `Phase Duration (min)` existen.  
Si algún grupo tiene más de un valor distinto, hay una inconsistencia.

In [108]:
# Valores distintos de Phase Duration (min) por cada Phase Id
duracion_por_fase = df.groupby("Phase Id")["Phase Duration (min)"].nunique()

# Fases con más de un valor distinto → inconsistencia
fases_inconsistentes_duracion = duracion_por_fase[duracion_por_fase > 1]

if fases_inconsistentes_duracion.empty:
    print("✅ Phase Duration (min) es consistente para todos los Phase Id.")
else:
    print(f"❌ Hay {len(fases_inconsistentes_duracion)} Phase Id con valores distintos de Phase Duration (min):\n")
    # Mostrar detalle de las fases problemáticas
    for phase_id in fases_inconsistentes_duracion.index:
        valores = df.loc[df["Phase Id"] == phase_id, "Phase Duration (min)"].unique()
        print(f"  Phase Id {phase_id}: valores = {valores}")

✅ Phase Duration (min) es consistente para todos los Phase Id.


### 2.2 ¿Es `Participated Players` consistente dentro de cada `Phase Id`?

Misma lógica: para cada `Phase Id`, el número de jugadores participantes debe ser único.

In [109]:
# Valores distintos de Participated Players por cada Phase Id
jugadores_por_fase = df.groupby("Phase Id")["Participated Players"].nunique()

# Fases con más de un valor distinto → inconsistencia
fases_inconsistentes_jugadores = jugadores_por_fase[jugadores_por_fase > 1]

if fases_inconsistentes_jugadores.empty:
    print("✅ Participated Players es consistente para todos los Phase Id.")
else:
    print(f"❌ Hay {len(fases_inconsistentes_jugadores)} Phase Id con valores distintos de Participated Players:\n")
    for phase_id in fases_inconsistentes_jugadores.index:
        valores = df.loc[df["Phase Id"] == phase_id, "Participated Players"].unique()
        print(f"  Phase Id {phase_id}: valores = {valores}")

✅ Participated Players es consistente para todos los Phase Id.


### 2.3 ¿Es `Player Participation Time (min)` el mismo para todos los jugadores dentro de cada `Phase Id`?

Si todos los jugadores de una misma fase tienen el mismo tiempo de participación, significa que
participaron el mismo tiempo. Si hay valores distintos, algunos jugadores participaron más o menos.

In [110]:
# Valores distintos de Player Participation Time (min) por cada Phase Id
participacion_por_fase = df.groupby("Phase Id")["Player Participation Time (min)"].nunique()

# Fases con más de un valor distinto → no todos participaron el mismo tiempo
fases_participacion_distinta = participacion_por_fase[participacion_por_fase > 1]

if fases_participacion_distinta.empty:
    print("✅ Player Participation Time (min) es el mismo para todos los jugadores en cada Phase Id.")
else:
    print(f"❌ Hay {len(fases_participacion_distinta)} Phase Id donde los jugadores tienen distinto Player Participation Time (min):\n")
    for phase_id in fases_participacion_distinta.head(15).index:
        valores = df.loc[df["Phase Id"] == phase_id, "Player Participation Time (min)"].unique()
        print(f"  Phase Id {phase_id}: valores = {valores}")
    if len(fases_participacion_distinta) > 15:
        print(f"  ... y {len(fases_participacion_distinta) - 15} Phase Id más.")

✅ Player Participation Time (min) es el mismo para todos los jugadores en cada Phase Id.


### 2.3 ¿Coinciden `Phase Duration (min)` y `Player Participation Time (min)` en cada fila?

Comprobamos fila a fila si la duración de la fase y el tiempo de participación del jugador son iguales.  
Si difieren, significa que el jugador **no participó durante toda la fase** (entró tarde, salió antes, etc.).

In [111]:
# Comparación fila a fila: Phase Duration vs Player Participation Time
df["duracion_coincide"] = df["Phase Duration (min)"] == df["Player Participation Time (min)"]

n_coinciden = df["duracion_coincide"].sum()
n_difieren = (~df["duracion_coincide"]).sum()
n_total = len(df)

print(f"Total de filas: {n_total}")
print(f"  ✅ Coinciden:  {n_coinciden} ({n_coinciden / n_total * 100:.1f}%)")
print(f"  ❌ Difieren:   {n_difieren} ({n_difieren / n_total * 100:.1f}%)")

if n_difieren > 0:
    print(f"\nMuestra de filas donde difieren:")
    cols_muestra = ["Phase Id", "Player Name", "Phase Duration (min)", "Player Participation Time (min)"]
    display(df.loc[~df["duracion_coincide"], cols_muestra].head(10))

# Limpieza: eliminamos la columna auxiliar
df.drop(columns=["duracion_coincide"], inplace=True)

Total de filas: 5321
  ✅ Coinciden:  2771 (52.1%)
  ❌ Difieren:   2550 (47.9%)

Muestra de filas donde difieren:


,Phase Id,Player Name,Phase Duration (min),Player Participation Time (min)
55,1656695,OLARRA 1489,11,NaN
61,1656695,MARTIN 1551,11,NaN
66,1656696,X X,2,3.0
67,1656696,A. MADINA 2648,2,3.0
68,1656696,NORES 2109,2,3.0
69,1656696,ETXEZARRETA 2230,2,3.0
70,1656696,GOROSABEL 1691,2,3.0
71,1656696,HODEI 2729,2,3.0
72,1656696,SARABIA 1561,2,3.0
73,1656696,NICOLAS 2694,2,3.0


In [112]:
# Desglose de los tipos de discrepancia
col_dur = "Phase Duration (min)"
col_part = "Player Participation Time (min)"

# Caso 1: Player Participation Time es NaN (el jugador no tiene dato de participación)
mask_nulo = df[col_part].isna()
n_nulos = mask_nulo.sum()

# Caso 2: Participación < Duración (el jugador participó menos tiempo que la fase)
mask_menor = (~mask_nulo) & (df[col_part] < df[col_dur])
n_menor = mask_menor.sum()

# Caso 3: Participación > Duración (el jugador aparece con más tiempo que la propia fase)
mask_mayor = (~mask_nulo) & (df[col_part] > df[col_dur])
n_mayor = mask_mayor.sum()

# Caso 4: Participación == Duración
mask_igual = (~mask_nulo) & (df[col_part] == df[col_dur])
n_igual = mask_igual.sum()

print("Desglose de la relación Phase Duration vs Player Participation Time:")
print(f"  ─ Participación == Duración (completa):     {n_igual:>5} ({n_igual/len(df)*100:.1f}%)")
print(f"  ─ Participación < Duración (parcial):       {n_menor:>5} ({n_menor/len(df)*100:.1f}%)")
print(f"  ─ Participación > Duración (⚠️ anómalo):    {n_mayor:>5} ({n_mayor/len(df)*100:.1f}%)")
print(f"  ─ Participación NaN (sin dato):             {n_nulos:>5} ({n_nulos/len(df)*100:.1f}%)")
print(f"  ─────────────────────────────────────────────")
print(f"  Total:                                      {len(df):>5}")

Desglose de la relación Phase Duration vs Player Participation Time:
  ─ Participación == Duración (completa):      2771 (52.1%)
  ─ Participación < Duración (parcial):           0 (0.0%)
  ─ Participación > Duración (⚠️ anómalo):     2374 (44.6%)
  ─ Participación NaN (sin dato):               176 (3.3%)
  ─────────────────────────────────────────────
  Total:                                       5321


In [113]:
# DataFrame con las filas donde Phase Duration != Player Participation Time
df_no_coinciden = df[df["Phase Duration (min)"] != df["Player Participation Time (min)"]].copy()

# Añadir columna con la diferencia para facilitar el análisis
df_no_coinciden["Diferencia (min)"] = (
    df_no_coinciden["Player Participation Time (min)"] - df_no_coinciden["Phase Duration (min)"]
)

print(f"Filas donde Phase Duration ≠ Player Participation Time: {len(df_no_coinciden)} de {len(df)}")
df_no_coinciden

Filas donde Phase Duration ≠ Player Participation Time: 2550 de 5321


,Session Id,Phase Id,Phase Duration (min),Participated Players,Player Name,Position Category,Position,Player Participation Time (min),Total Touches (#),Left Leg Touches (#),...,Release Velocity Max (m/s),Release Velocity Avg (m/s),Release Index Beta - Total,Player Id,Agrupacion,Espacio,Polaridad,Equilibrio,NombreCorrecto,Diferencia (min)
55,1656564,1656695,11,22,OLARRA 1489,Defenders,Left Back,NaN,0.0,0.0,...,NaN,NaN,0.0,114620,AMedia,EAmplio,Polarizado,Equilibrio,Juvenil División de Honor,NaN
61,1656564,1656695,11,22,MARTIN 1551,Defenders,Right Center Back,NaN,0.0,0.0,...,NaN,NaN,0.0,114636,AMedia,EAmplio,Polarizado,Equilibrio,Juvenil División de Honor,NaN
66,1656564,1656696,2,22,X X,Midfielders,Defensive Midfielder,3.0,6.0,1.0,...,19.4,19.4,1.9,111224,APequena,EAmplio,Polarizado,Equilibrio,Juvenil División de Honor,1.0
67,1656564,1656696,2,22,A. MADINA 2648,Forwards,Right Winger,3.0,15.0,3.0,...,18.0,18.0,1.8,113152,APequena,EAmplio,Polarizado,Equilibrio,Juvenil División de Honor,1.0
68,1656564,1656696,2,22,NORES 2109,Defenders,Right Center Back,3.0,0.0,0.0,...,NaN,NaN,0.0,113153,APequena,EAmplio,Polarizado,Equilibrio,Juvenil División de Honor,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5316,1479037,1490663,12,20,MIDJE 2094,Defenders,Right Back,13.0,5.0,1.0,...,NaN,NaN,0.0,114715,AGrande,EMedio,Polarizado,Equilibrio,Neskak C,1.0
5317,1479037,1490663,12,20,ISASISASMENDI 1984,Midfielders,Center Midfielder,13.0,30.0,7.0,...,18.3,11.0,7.7,114716,AGrande,EMedio,Polarizado,Equilibrio,Neskak C,1.0
5318,1479037,1490663,12,20,NAHIA SARRIEGI 2096,Forwards,Striker,13.0,10.0,4.0,...,NaN,NaN,0.0,114717,AGrande,EMedio,Polarizado,Equilibrio,Neskak C,1.0
5319,1479037,1490663,12,20,MARCOS 1975,Defenders,Left Back,13.0,17.0,2.0,...,22.8,13.5,12.1,114718,AGrande,EMedio,Polarizado,Equilibrio,Neskak C,1.0


In [114]:
# Filas con Player Participation Time = NaN
df_nan_participacion = df[df["Player Participation Time (min)"].isna()]

# Verificar que todas tienen Total Touches = 0 o NaN
todos_touches_cero_o_nan = df_nan_participacion["Total Touches (#)"].apply(lambda x: x == 0 or pd.isna(x)).all()
print(f"Filas con Player Participation Time = NaN: {len(df_nan_participacion)}")
print(f"¿Todas tienen Total Touches (#) = 0 o NaN? → {'✅ Sí' if todos_touches_cero_o_nan else '❌ No'}")

# Excluir esas filas del df base (no aportan información útil)
n_antes = len(df)
df = df.dropna(subset=["Player Participation Time (min)"])
print(f"\ndf antes: {n_antes} filas → df después: {len(df)} filas (eliminadas {n_antes - len(df)})")

# Actualizar también df_no_coinciden
df_no_coinciden = df_no_coinciden.dropna(subset=["Player Participation Time (min)"])
print(f"df_no_coinciden tras excluir NaN: {len(df_no_coinciden)} filas")

Filas con Player Participation Time = NaN: 176
¿Todas tienen Total Touches (#) = 0 o NaN? → ✅ Sí

df antes: 5321 filas → df después: 5145 filas (eliminadas 176)
df_no_coinciden tras excluir NaN: 2374 filas


In [115]:
# ¿Hay filas con Player Participation Time (min) = 0 o Phase Duration (min) = 0?
n_participacion_cero = (df["Player Participation Time (min)"] == 0).sum()
n_duracion_cero = (df["Phase Duration (min)"] == 0).sum()

print(f"Filas con Player Participation Time (min) = 0: {n_participacion_cero}")
print(f"Filas con Phase Duration (min) = 0: {n_duracion_cero}")

if n_participacion_cero > 0:
    print(f"\nMuestra de filas con Player Participation Time = 0:")
    display(df[df["Player Participation Time (min)"] == 0].head(10))

if n_duracion_cero > 0:
    print(f"\nMuestra de filas con Phase Duration = 0:")
    display(df[df["Phase Duration (min)"] == 0].head(10))

Filas con Player Participation Time (min) = 0: 0
Filas con Phase Duration (min) = 0: 0


In [116]:
# Distribución de la diferencia (min) entre Player Participation Time y Phase Duration
print("Value counts de Diferencia (min) en df_no_coinciden:\n")
df_no_coinciden["Diferencia (min)"].value_counts().sort_index()

Value counts de Diferencia (min) en df_no_coinciden:



Diferencia (min)
1.0    2374
Name: count, dtype: int64

Todas las 2374 filas que difieren tienen exactamente una diferencia de +1 minuto. Es decir, en todos los casos Player Participation Time = Phase Duration + 1. Esto apunta a un redondeo sistemático en la plataforma PlayerMaker, no a datos sucios puntuales.

---

### 📋 Resumen hasta aquí

**Comprobaciones realizadas (2.1 – 2.3):**

| Comprobación | Resultado |
|---|---|
| `Phase Duration (min)` consistente por `Phase Id` | ✅ OK |
| `Participated Players` consistente por `Phase Id` | ✅ OK |
| `Player Participation Time (min)` igual para todos los jugadores en cada `Phase Id` | ✅ OK |
| `Phase Duration` vs `Player Participation Time` por fila | ⚠️ 2374 filas difieren en +1 min (redondeo sistemático) |
| Filas con `Player Participation Time = 0` o `Phase Duration = 0` | ✅ No hay ninguna |

**Cambio aplicado al `df` base:**

- **Eliminadas 176 filas** con `Player Participation Time (min) = NaN` (todas tenían `Total Touches = 0` o `NaN` → sin actividad registrada).
- **`df` pasa de 5321 → 5145 filas.** Las comprobaciones a partir de aquí trabajan sobre este `df` limpio.

---

### 2.4 ¿Son `Agrupacion`, `Espacio`, `Polaridad` y `Equilibrio` consistentes dentro de cada `Phase Id`?

Estas cuatro columnas categóricas describen el tipo de ejercicio/fase. Deberían tener un único valor
para cada `Phase Id`, ya que son atributos de la fase, no del jugador.

In [117]:
# Columnas categóricas que deben ser constantes por Phase Id
cols_categoricas = ["Agrupacion", "Espacio", "Polaridad", "Equilibrio"]

todo_ok = True
for col in cols_categoricas:
    valores_unicos_por_fase = df.groupby("Phase Id")[col].nunique()
    fases_inconsistentes = valores_unicos_por_fase[valores_unicos_por_fase > 1]

    if fases_inconsistentes.empty:
        print(f"✅ {col} — consistente para todos los Phase Id.")
    else:
        todo_ok = False
        print(f"❌ {col} — {len(fases_inconsistentes)} Phase Id con valores distintos:")
        for phase_id in fases_inconsistentes.index:
            valores = df.loc[df["Phase Id"] == phase_id, col].unique()
            print(f"     Phase Id {phase_id}: {valores}")
    print()

if todo_ok:
    print("─" * 50)
    print("Todas las columnas categóricas de fase son consistentes.")

✅ Agrupacion — consistente para todos los Phase Id.

✅ Espacio — consistente para todos los Phase Id.

✅ Polaridad — consistente para todos los Phase Id.

✅ Equilibrio — consistente para todos los Phase Id.

──────────────────────────────────────────────────
Todas las columnas categóricas de fase son consistentes.


### 2.5 ¿Hay algún `Player Id` repetido dentro del mismo `Phase Id`?

Un jugador no debería aparecer más de una vez en la misma fase. Si hay duplicados,
puede tratarse de filas duplicadas o de un error de registro.

In [118]:
# Filas duplicadas por combinación (Phase Id, Player Id)
duplicados_player_id = df[df.duplicated(subset=["Phase Id", "Player Id"], keep=False)]

n_filas_dup = len(duplicados_player_id)
n_combinaciones_dup = duplicados_player_id.groupby(["Phase Id", "Player Id"]).ngroups if n_filas_dup > 0 else 0

if n_filas_dup == 0:
    print("✅ No hay Player Id repetido dentro del mismo Phase Id.")
else:
    print(f"❌ Hay {n_filas_dup} filas implicadas en {n_combinaciones_dup} combinaciones (Phase Id, Player Id) duplicadas.\n")
    cols_muestra = ["Phase Id", "Player Id", "Player Name", "Phase Duration (min)", "Player Participation Time (min)"]
    display(duplicados_player_id.sort_values(["Phase Id", "Player Id"])[cols_muestra].head(20))

❌ Hay 120 filas implicadas en 60 combinaciones (Phase Id, Player Id) duplicadas.



,Phase Id,Player Id,Player Name,Phase Duration (min),Player Participation Time (min)
3164,1456231,114673,GORROTXATEGI 13,13,14.0
3224,1456231,114673,GORROTXATEGI 13,13,14.0
3165,1456231,114674,DADIE 1358,13,14.0
3225,1456231,114674,DADIE 1358,13,14.0
3166,1456231,114675,CARBONELL 1281,13,14.0
3226,1456231,114675,CARBONELL 1281,13,14.0
3167,1456231,114677,ZOILO 1253,13,14.0
3227,1456231,114677,ZOILO 1253,13,14.0
3168,1456231,114678,ARAMBARRI 1197,13,14.0
3228,1456231,114678,ARAMBARRI 1197,13,14.0


In [119]:
# ¿Son duplicados completos (todas las columnas iguales)?
n_dup_completos = df.duplicated(keep=False).sum()
n_dup_por_phase_player = len(duplicados_player_id)

# Comprobar si cada par duplicado (Phase Id, Player Id) tiene filas idénticas en TODAS las columnas
son_dup_completos = True
for (phase, player), grupo in duplicados_player_id.groupby(["Phase Id", "Player Id"]):
    # Si eliminamos duplicados completos y queda más de 1 fila, no son idénticas
    if len(grupo.drop_duplicates()) > 1:
        son_dup_completos = False
        print(f"  ⚠️ Phase Id {phase}, Player Id {player}: filas NO son idénticas")
        display(grupo)

if son_dup_completos:
    print(f"✅ Todas las {n_combinaciones_dup} combinaciones duplicadas (Phase Id, Player Id) son filas completamente idénticas.")
    
    # Eliminar duplicados completos del df base (quedarnos con la primera ocurrencia)
    n_antes = len(df)
    df = df.drop_duplicates(keep="first")
    n_eliminadas = n_antes - len(df)
    print(f"\n🗑️ Eliminados {n_eliminadas} duplicados completos del df base.")
    print(f"   df antes: {n_antes} filas → df después: {len(df)} filas")

✅ Todas las 60 combinaciones duplicadas (Phase Id, Player Id) son filas completamente idénticas.

🗑️ Eliminados 60 duplicados completos del df base.
   df antes: 5145 filas → df después: 5085 filas


### 2.5 ¿Hay algún `Player Name` repetido dentro del mismo `Phase Id`?

Comprobación análoga a la anterior pero usando el nombre del jugador en lugar de su identificador numérico.

In [120]:
# Filas duplicadas por combinación (Phase Id, Player Name)
duplicados_player_name = df[df.duplicated(subset=["Phase Id", "Player Name"], keep=False)]

n_filas_dup_name = len(duplicados_player_name)
n_combinaciones_dup_name = duplicados_player_name.groupby(["Phase Id", "Player Name"]).ngroups if n_filas_dup_name > 0 else 0

if n_filas_dup_name == 0:
    print("✅ No hay Player Name repetido dentro del mismo Phase Id.")
else:
    print(f"❌ Hay {n_filas_dup_name} filas implicadas en {n_combinaciones_dup_name} combinaciones (Phase Id, Player Name) duplicadas.\n")
    cols_muestra = ["Phase Id", "Player Id", "Player Name", "Phase Duration (min)", "Player Participation Time (min)"]
    display(duplicados_player_name.sort_values(["Phase Id", "Player Name"])[cols_muestra].head(20))

❌ Hay 37 filas implicadas en 14 combinaciones (Phase Id, Player Name) duplicadas.



,Phase Id,Player Id,Player Name,Phase Duration (min),Player Participation Time (min)
2754,1497222,111224,X X,22,22.0
2755,1497222,111225,X X,22,22.0
2756,1497222,111226,X X,22,22.0
2773,1497223,111224,X X,30,30.0
2774,1497223,111225,X X,30,30.0
2775,1497223,111226,X X,30,30.0
2792,1497224,111224,X X,20,20.0
2793,1497224,111225,X X,20,20.0
2794,1497224,111226,X X,20,20.0
2809,1501396,111224,X X,31,31.0


In [121]:
# Resumen de los duplicados por Player Name que NO son idénticos
# ¿Cuántos Player Id distintos hay en cada grupo?
print("Detalle de los duplicados por Player Name no idénticos:\n")
print(f"{'Phase Id':<12} {'Player Name':<10} {'N filas':<10} {'Player Ids distintos'}")
print("-" * 60)

n_filas_no_identicas = 0
for (phase, pname), grupo in duplicados_player_name.groupby(["Phase Id", "Player Name"]):
    if len(grupo.drop_duplicates()) > 1:
        ids_distintos = grupo["Player Id"].unique()
        n_filas_no_identicas += len(grupo)
        print(f"{phase:<12} {pname:<10} {len(grupo):<10} {ids_distintos}")

print(f"\n📊 Total: {n_filas_no_identicas} filas en {14} combinaciones (Phase Id, Player Name)")
print(f"   Causa: jugadores distintos (diferentes Player Id) que comparten el mismo nombre anonimizado.")
print(f"   ➜ NO se eliminan del df, son datos legítimos de jugadores diferentes.")
print(f"\n   df se mantiene en {len(df)} filas.")

Detalle de los duplicados por Player Name no idénticos:

Phase Id     Player Name N filas    Player Ids distintos
------------------------------------------------------------
1497222      X X        3          [111224 111225 111226]
1497223      X X        3          [111224 111225 111226]
1497224      X X        3          [111224 111225 111226]
1501396      X X        3          [111224 111225 111226]
1501397      X X        3          [111224 111225 111226]
1501467      X X        2          [111224 111226]
1501469      X X        2          [111224 111226]
1518331      X X        2          [111225 111226]
1518332      X X        2          [111225 111226]
1518333      X X        2          [111225 111226]
1522739      X X        3          [111224 111225 111226]
1522740      X X        3          [111224 111225 111226]
1522882      X X        3          [111224 111225 111226]
1522884      X X        3          [111224 111225 111226]

📊 Total: 37 filas en 14 combinaciones (Phase Id

**Conclusión sección 2.5:** Los 14 casos de duplicados por `Player Name` dentro del mismo `Phase Id` corresponden **todos** al nombre anonimizado `'X X'`, que agrupa a **3 jugadores reales distintos** con `Player Id` 111224, 111225 y 111226 (posiciones: Defensive Midfielder, Goalkeeper y Striker respectivamente). 

No son duplicados reales sino jugadores diferentes cuyo nombre fue anonimizado de la misma forma en PlayerMaker. **No se elimina ninguna fila** — el df se mantiene en **5 085 filas**.

### 2.7 ¿Es la relación `Player Id` ↔ `Player Name` consistente?

Comprobamos en ambas direcciones:
1. **Un `Player Id` → ¿siempre el mismo `Player Name`?** (si no, hay nombres inconsistentes para el mismo jugador)
2. **Un `Player Name` → ¿siempre el mismo `Player Id`?** (si no, hay jugadores distintos con el mismo nombre)

In [122]:
# --- Dirección 1: ¿Un mismo Player Id tiene siempre el mismo Player Name? ---
nombres_por_id = df.groupby("Player Id")["Player Name"].nunique()
ids_con_varios_nombres = nombres_por_id[nombres_por_id > 1]

if ids_con_varios_nombres.empty:
    print("✅ Cada Player Id tiene un único Player Name.")
else:
    print(f"❌ Hay {len(ids_con_varios_nombres)} Player Id con más de un Player Name:\n")
    for pid in ids_con_varios_nombres.index:
        nombres = df.loc[df["Player Id"] == pid, "Player Name"].unique()
        n_filas = len(df[df["Player Id"] == pid])
        print(f"  Player Id {pid} ({n_filas} filas): {list(nombres)}")

print("\n" + "─" * 60 + "\n")

# --- Dirección 2: ¿Un mismo Player Name tiene siempre el mismo Player Id? ---
ids_por_nombre = df.groupby("Player Name")["Player Id"].nunique()
nombres_con_varios_ids = ids_por_nombre[ids_por_nombre > 1]

if nombres_con_varios_ids.empty:
    print("✅ Cada Player Name tiene un único Player Id.")
else:
    print(f"❌ Hay {len(nombres_con_varios_ids)} Player Name con más de un Player Id:\n")
    for pname in nombres_con_varios_ids.index:
        ids = df.loc[df["Player Name"] == pname, "Player Id"].unique()
        n_filas = len(df[df["Player Name"] == pname])
        print(f"  Player Name '{pname}' ({n_filas} filas): Player Ids = {list(ids)}")

✅ Cada Player Id tiene un único Player Name.

────────────────────────────────────────────────────────────

❌ Hay 29 Player Name con más de un Player Id:

  Player Name 'AGUIRRE 1704' (16 filas): Player Ids = [np.int64(114570), np.int64(113190)]
  Player Name 'ARANA 1361' (30 filas): Player Ids = [np.int64(114681), np.int64(114221)]
  Player Name 'AZPIROTZ 2728' (18 filas): Player Ids = [np.int64(111243), np.int64(114719)]
  Player Name 'BARO 1705' (36 filas): Player Ids = [np.int64(114572), np.int64(113168)]
  Player Name 'CHAVEZ 1709' (26 filas): Player Ids = [np.int64(114573), np.int64(113202)]
  Player Name 'CORENGIA 1707' (29 filas): Player Ids = [np.int64(114571), np.int64(113178)]
  Player Name 'ELCANO 1711' (32 filas): Player Ids = [np.int64(114574), np.int64(113198)]
  Player Name 'GONZÁLEZ DE ZÁRATE 1426' (19 filas): Player Ids = [np.int64(114695), np.int64(115445)]


  Player Name 'HARUTO 2663' (39 filas): Player Ids = [np.int64(114567), np.int64(113188)]
  Player Name 'ISASISASMENDI 1984' (22 filas): Player Ids = [np.int64(111229), np.int64(114716)]
  Player Name 'KOYATA 2641' (30 filas): Player Ids = [np.int64(114566), np.int64(113196)]
  Player Name 'LARRALDE 2234' (46 filas): Player Ids = [np.int64(114584), np.int64(113174)]
  Player Name 'LÓPEZ 2054' (29 filas): Player Ids = [np.int64(114581), np.int64(113182)]
  Player Name 'MARCHAL 2706' (27 filas): Player Ids = [np.int64(114586), np.int64(113179)]
  Player Name 'MARCOS 1975' (18 filas): Player Ids = [np.int64(111249), np.int64(114718)]
  Player Name 'MARIEZKURRENA 1493' (39 filas): Player Ids = [np.int64(114622), np.int64(111223)]
  Player Name 'MARTÍN 1599' (25 filas): Player Ids = [np.int64(114676), np.int64(114210)]
  Player Name 'NAHIA SARRIEGI 2096' (20 filas): Player Ids = [np.int64(111244), np.int64(114717)]
  Player Name 'OIER IPINTZA 2590' (29 filas): Player Ids = [np.int64(114197)

In [123]:
# --- Unificación de Player Id para Player Names con múltiples Ids ---
# Excluimos 'X X' porque sus 3 Ids son jugadores realmente distintos
# (coexisten en las mismas fases con posiciones diferentes)
# Para el resto: nunca coexisten en la misma fase → son re-registros del mismo jugador

nombres_a_unificar = [n for n in nombres_con_varios_ids.index if n != "X X"]

n_reemplazos = 0
print("Aplicando unificación de Player Id...\n")

for pname in sorted(nombres_a_unificar):
    sub = df[df["Player Name"] == pname]
    conteo_ids = sub["Player Id"].value_counts()
    id_mayor = conteo_ids.index[0]       # Id con más filas → se mantiene
    ids_menores = conteo_ids.index[1:]    # Ids minoritarios → se reemplazan
    
    for id_min in ids_menores:
        mask = (df["Player Name"] == pname) & (df["Player Id"] == id_min)
        n = mask.sum()
        if n > 0:
            df.loc[mask, "Player Id"] = id_mayor
            n_reemplazos += n
            print(f"  {pname:<30} Player Id {id_min} → {id_mayor}  ({n} filas)")

print(f"\n✅ Unificación completada: {n_reemplazos} filas actualizadas en {len(nombres_a_unificar)} jugadores.")
print(f"   df mantiene {len(df)} filas (no se eliminó ninguna, solo se cambió el Player Id).")

# Verificación post-unificación
ids_por_nombre_post = df.groupby("Player Name")["Player Id"].nunique()
nombres_varios_ids_post = ids_por_nombre_post[ids_por_nombre_post > 1]
print(f"\n--- Verificación post-unificación ---")
print(f"   Player Names con múltiples Player Id: {len(nombres_varios_ids_post)}")
if not nombres_varios_ids_post.empty:
    for pname in nombres_varios_ids_post.index:
        ids = df.loc[df["Player Name"] == pname, "Player Id"].unique()
        print(f"   • '{pname}': {list(ids)}")
    print("   (Solo debería quedar 'X X', que tiene 3 jugadores reales distintos)")
else:
    print("   ✅ Todos los Player Name tienen ahora un único Player Id.")

Aplicando unificación de Player Id...

  AGUIRRE 1704                   Player Id 113190 → 114570  (7 filas)
  ARANA 1361                     Player Id 114221 → 114681  (2 filas)
  AZPIROTZ 2728                  Player Id 114719 → 111243  (7 filas)
  BARO 1705                      Player Id 114572 → 113168  (9 filas)
  CHAVEZ 1709                    Player Id 114573 → 113202  (6 filas)
  CORENGIA 1707                  Player Id 113178 → 114571  (10 filas)
  ELCANO 1711                    Player Id 114574 → 113198  (12 filas)
  GONZÁLEZ DE ZÁRATE 1426        Player Id 115445 → 114695  (3 filas)
  HARUTO 2663                    Player Id 114567 → 113188  (19 filas)
  ISASISASMENDI 1984             Player Id 114716 → 111229  (11 filas)
  KOYATA 2641                    Player Id 114566 → 113196  (10 filas)
  LARRALDE 2234                  Player Id 114584 → 113174  (17 filas)
  LÓPEZ 2054                     Player Id 113182 → 114581  (10 filas)
  MARCHAL 2706                   Player Id 1

In [124]:
# --- Eliminación de columnas redundantes ---
# - Player Name: ya tenemos NombreCorrecto como referencia y Player Id unificado
# - Player Participation Time (min): coincide con Phase Duration salvo redondeo de +1 min

cols_eliminar = ["Player Name", "Player Participation Time (min)"]
print(f"Columnas antes: {len(df.columns)}")
df.drop(columns=cols_eliminar, inplace=True)
print(f"Columnas después: {len(df.columns)}")
print(f"\n🗑️ Eliminadas: {cols_eliminar}")
print(f"   Columnas restantes ({len(df.columns)}):\n   {list(df.columns)}")

Columnas antes: 30
Columnas después: 28

🗑️ Eliminadas: ['Player Name', 'Player Participation Time (min)']
   Columnas restantes (28):
   ['Session Id', 'Phase Id', 'Phase Duration (min)', 'Participated Players', 'Position Category', 'Position', 'Total Touches (#)', 'Left Leg Touches (#)', 'Right Leg Touches (#)', 'Releases (#)', 'RV Zone 1 [0-5( m/s)]', 'RV Zone 2 [5-10( m/s)]', 'RV Zone 3 [10-15( m/s)]', 'RV Zone 4 [15-20( m/s)]', 'RV Zone 5 [20-25( m/s)]', 'RV Zone 6 [> 25( m/s)]', 'Distance Covered (m)', 'HID Covered (m) Zone 1 [> 4(m/s)]', 'SD Covered (m) [> 5.5(m/s)]', 'Release Velocity Max (m/s)', 'Release Velocity Avg (m/s)', 'Release Index Beta - Total', 'Player Id', 'Agrupacion', 'Espacio', 'Polaridad', 'Equilibrio', 'NombreCorrecto']


### 2.8 Reclasificación de `Agrupacion` y `Espacio`

Las columnas originales `Agrupacion` (APequena, AMedia, AGrande) y `Espacio` (EReducido, EMedio, EAmplio) tienen **7 combinaciones válidas** que se recodifican en categorías simplificadas. Las combinaciones que no encajan en ninguna categoría se eliminan del dataset.

**Mapeo aplicado:**

| Agrupacion | Espacio | → espacio × agrupación |
|---|---|---|
| APequena / AMedia | EReducido | reducido-pequeno |
| AGrande | EAmplio / EMedio | amplio-grande |
| APequena / AMedia | EAmplio | amplio-pequeno |
| AGrande | EReducido | reducido-grande |

Tras el mapeo, las columnas `Agrupacion` y `Espacio` se actualizan con los componentes simplificados:
- `Agrupacion` → `pequeno` o `grande`
- `Espacio` → `reducido` o `amplio`

In [125]:
# --- 2.8 Reclasificación de Agrupacion y Espacio ---

# Mapeo de combinaciones válidas
mapeo_agrupacion_espacio = {
    ('APequena', 'EReducido'): 'reducido-pequeno',
    ('AMedia',   'EReducido'): 'reducido-pequeno',
    ('AGrande',  'EAmplio'):   'amplio-grande',
    ('AGrande',  'EMedio'):    'amplio-grande',
    ('APequena', 'EAmplio'):   'amplio-pequeno',
    ('AMedia',   'EAmplio'):   'amplio-pequeno',
    ('AGrande',  'EReducido'): 'reducido-grande',
}

# Crear columna combinada
df['espacio_x_agrupacion'] = df.apply(
    lambda row: mapeo_agrupacion_espacio.get(
        (row['Agrupacion'], row['Espacio']),
        'sin_clasificar'
    ),
    axis=1
)

In [126]:
# Diagnóstico: distribución de la variable combinada
print("Distribución de espacio × agrupación:\n")
vc = df['espacio_x_agrupacion'].value_counts()
for cat, n in vc.items():
    print(f"  {cat:<25} {n:>5} ({n/len(df)*100:.1f}%)")

n_sin = (df['espacio_x_agrupacion'] == 'sin_clasificar').sum()
print(f"\n{'─'*50}")
print(f"  sin_clasificar: {n_sin} filas ({n_sin/len(df)*100:.1f}%)")

# Mostrar las combinaciones que quedan sin clasificar
if n_sin > 0:
    combos_sin = df.loc[df['espacio_x_agrupacion'] == 'sin_clasificar', ['Agrupacion', 'Espacio']].drop_duplicates()
    print(f"\n  Combinaciones sin clasificar:")
    for _, row in combos_sin.iterrows():
        n_combo = ((df['Agrupacion'] == row['Agrupacion']) & (df['Espacio'] == row['Espacio'])).sum()
        print(f"    ({row['Agrupacion']}, {row['Espacio']}) → {n_combo} filas")

Distribución de espacio × agrupación:

  reducido-pequeno           1722 (33.9%)
  amplio-grande              1704 (33.5%)
  reducido-grande             660 (13.0%)
  amplio-pequeno              535 (10.5%)
  sin_clasificar              464 (9.1%)

──────────────────────────────────────────────────
  sin_clasificar: 464 filas (9.1%)

  Combinaciones sin clasificar:
    (AMedia, EMedio) → 190 filas
    (APequena, EMedio) → 274 filas


In [127]:
# --- Eliminar filas sin clasificar y recodificar Agrupacion / Espacio ---

n_antes = len(df)
df = df[df['espacio_x_agrupacion'] != 'sin_clasificar'].copy()
n_eliminadas_sc = n_antes - len(df)
print(f"🗑️ Eliminadas {n_eliminadas_sc} filas sin clasificar")
print(f"   df: {n_antes} → {len(df)} filas\n")

# Extraer componentes de la categoría combinada (formato: "espacio-agrupacion")
df['Espacio']     = df['espacio_x_agrupacion'].str.split('-').str[0]
df['Agrupacion']  = df['espacio_x_agrupacion'].str.split('-').str[1]

# Eliminar columna temporal
df.drop(columns=['espacio_x_agrupacion'], inplace=True)

# Verificación
print("Nuevos valores de Agrupacion:")
for val, n in df['Agrupacion'].value_counts().items():
    print(f"  {val:<15} {n:>5} ({n/len(df)*100:.1f}%)")

print(f"\nNuevos valores de Espacio:")
for val, n in df['Espacio'].value_counts().items():
    print(f"  {val:<15} {n:>5} ({n/len(df)*100:.1f}%)")

print(f"\n✅ Reclasificación completada — df: {len(df)} filas × {df.shape[1]} columnas")

🗑️ Eliminadas 464 filas sin clasificar
   df: 5085 → 4621 filas

Nuevos valores de Agrupacion:
  grande           2364 (51.2%)
  pequeno          2257 (48.8%)

Nuevos valores de Espacio:
  reducido         2382 (51.5%)
  amplio           2239 (48.5%)

✅ Reclasificación completada — df: 4621 filas × 28 columnas


In [128]:
# --- Exportación del df limpio ---
ruta_salida = "../Datos/Matriz_V1.xlsx"
df.to_excel(ruta_salida, index=False)
print(f"✅ df exportado correctamente a: {ruta_salida}")
print(f"   Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

✅ df exportado correctamente a: ../Datos/Matriz_V1.xlsx
   Dimensiones: 4621 filas × 28 columnas


---

## 📋 Resumen final de la limpieza previa

### Dataset de partida

| Concepto | Valor |
|---|---|
| **Archivo fuente** | `PlayerMaker_Matrix.xlsx` |
| **Filas originales** | 5 321 |
| **Columnas originales** | 30 |

### Pipeline de transformaciones

| Paso | Sección | Acción | Filas eliminadas | Filas modificadas | Columnas eliminadas | df resultante |
|:----:|:-------:|--------|:----------------:|:-----------------:|:-------------------:|:-------------:|
| 1 | 2.3 | Eliminar filas con `Player Participation Time = NaN` (todas tenían `Total Touches = 0` o `NaN` → sin actividad registrada) | **176** | — | — | 5 145 × 30 |
| 2 | 2.5 | Eliminar duplicados completos por `(Phase Id, Player Id)` — 60 pares de filas idénticas en todas las columnas | **60** | — | — | 5 085 × 30 |
| 3 | 2.7 | Unificar `Player Id` para 28 jugadores re-registrados (cada `Player Name` con múltiples Ids que nunca coexisten en la misma fase → mismo jugador) | — | **293** | — | 5 085 × 30 |
| 4 | — | Eliminar columnas `Player Name` (redundante con `Player Id`) y `Player Participation Time (min)` (redundante con `Phase Duration`, salvo redondeo sistemático de +1 min) | — | — | **2** | 5 085 × 28 |
| 5 | 2.8 | Reclasificación de `Agrupacion` × `Espacio`: mapeo de 7 combinaciones válidas → 4 categorías cruzadas. Eliminación de 464 filas sin clasificar (combinaciones `AMedia×EMedio` y `APequena×EMedio`). Recodificación de `Agrupacion` → `pequeno` / `grande` y `Espacio` → `reducido` / `amplio` | **464** | **4 621** | — | **4 621 × 28** |

### Comprobaciones de consistencia superadas ✅

| Comprobación | Sección | Resultado |
|---|:---:|---|
| `Phase Duration (min)` consistente por `Phase Id` | 2.1 | ✅ OK |
| `Participated Players` consistente por `Phase Id` | 2.2 | ✅ OK |
| `Player Participation Time (min)` igual para todos los jugadores en cada `Phase Id` | 2.3 | ✅ OK |
| `Agrupacion`, `Espacio`, `Polaridad`, `Equilibrio` consistentes por `Phase Id` | 2.4 | ✅ OK |
| Un `Player Id` → un único `Player Name` | 2.7 | ✅ OK |
| Un `Player Name` → un único `Player Id` (post-unificación) | 2.7 | ✅ OK (excepto `'X X'`: 3 jugadores reales distintos) |

### Reclasificación de `Agrupacion` y `Espacio` (sección 2.8)

| Agrupacion original | Espacio original | → Categoría combinada | Agrupacion final | Espacio final |
|---|---|---|---|---|
| APequena / AMedia | EReducido | reducido-pequeno (1 722) | pequeno | reducido |
| AGrande | EAmplio / EMedio | amplio-grande (1 704) | grande | amplio |
| APequena / AMedia | EAmplio | amplio-pequeno (535) | pequeno | amplio |
| AGrande | EReducido | reducido-grande (660) | grande | reducido |
| AMedia / APequena | EMedio | sin_clasificar (464) | — | *eliminadas* |

### Observaciones registradas (sin acción correctiva)

- **2 374 filas** donde `Player Participation Time = Phase Duration + 1 min` → redondeo sistemático de la plataforma PlayerMaker, no datos sucios.
- **`'X X'`** (57 filas, 3 `Player Id` distintos: 111224, 111225, 111226) → jugadores reales diferentes con nombre anonimizado. Coexisten en las mismas fases con posiciones distintas (Defensive Midfielder, Goalkeeper, Striker). Se mantienen sin modificar.
- **`'Player 1'`, `'Player 22'`, `'Player 23'`, `'Player 24'`** → nombres genéricos con datos de rendimiento válidos. Se mantienen.

### Estado final del `df` → exportado como `Datos/Matriz_V1.xlsx`

| Concepto | Valor |
|---|---|
| **Filas** | **4 621** |
| **Columnas** | **28** |
| **Filas eliminadas en total** | 700 (13.2% del original) |
| **Columnas eliminadas** | 2 (`Player Name`, `Player Participation Time (min)`) |
| **Filas con Player Id modificado** | 293 |
| **Archivo de salida** | `Datos/Matriz_V1.xlsx` |